In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

#Membaca dataset
df = pd.read_csv(r"B:\Python\DataSet\CSV\Zomato Restaurant Dataset.csv")
#print(df.columns.tolist())
'''
['Restaurant ID', 'Restaurant Name', 'Country Code',
'City', 'Address', 'Locality', 
'Locality Verbose', 'Longitude', 'Latitude', 
'Cuisines', 'Average Cost for two', 'Currency', 
'Has Table booking', 'Has Online delivery', 'Is delivering now', 
'Switch to order menu', 'Price range', 'Aggregate rating', 
'Rating color', 'Rating text', 'Votes']
'''

"\n['Restaurant ID', 'Restaurant Name', 'Country Code',\n'City', 'Address', 'Locality', \n'Locality Verbose', 'Longitude', 'Latitude', \n'Cuisines', 'Average Cost for two', 'Currency', \n'Has Table booking', 'Has Online delivery', 'Is delivering now', \n'Switch to order menu', 'Price range', 'Aggregate rating', \n'Rating color', 'Rating text', 'Votes']\n"

In [5]:
# Memeriksa distribusi biaya rata-rata untuk dua orang
df["Average Cost for two"].describe()

count      9551.000000
mean       1199.210763
std       16121.183073
min           0.000000
25%         250.000000
50%         400.000000
75%         700.000000
max      800000.000000
Name: Average Cost for two, dtype: float64

In [13]:
hasil = (
    df.groupby("Currency")["Average Cost for two"]
    .agg(
            jumlah="count",
            rata="mean",
            median="median",
            std="std",
            min="min",
            l = lambda x: np.percentile(x, 25) - 1.5 *(np.percentile(x,75) - np.percentile(x, 25)),
            max="max",
            u = lambda x: np.percentile(x, 75) + 1.5 *(np.percentile(x,75) - np.percentile(x, 25))
    )
)
print(hasil)

                        jumlah           rata    median            std    min  \
Currency                                                                        
Botswana Pula(P)            22    1606.818182    1150.0    1268.510136    600   
Brazilian Real(R$)          60     134.666667     100.0      86.831547     30   
Dollar($)                  482      31.510373      25.0      41.879355      0   
Emirati Diram(AED)          60     166.416667     145.0     109.969532     40   
Indian Rupees(Rs.)        8652     623.370319     450.0     595.676124      0   
Indonesian Rupiah(IDR)      21  281190.476190  200000.0  203543.022245  70000   
NewZealand($)               40      69.750000      60.0      37.824663     20   
Pounds(��)                  80      47.812500      40.0      34.520621     10   
Qatari Rial(QR)             20     223.750000     185.0     147.905331     50   
Rand(R)                     60     419.733333     340.0     422.196069    110   
Sri Lankan Rupee(LKR)       

In [14]:
# Menghitung kuartil per kelompok mata uang
groups = df.groupby("Currency")["Average Cost for two"]

q1 = groups.transform(lambda s: s.quantile(0.25))
q3 = groups.transform(lambda s: s.quantile(0.75))

# Menghitung IQR dan batas setiap baris
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

# Menandai kandidat outlier
cost = df["Average Cost for two"]

df["is_outlier"] = (cost < lower) | (cost > upper)

# Ringkasan per mata uang
summary = df.groupby("Currency").agg(
    jumlah=("Average Cost for two", "size"),
    kandidat_outlier=("is_outlier", "sum"),
    persentase_outlier=("is_outlier", lambda s: s.mean() * 100)
)

summary["persentase_outlier"] = (
    summary["persentase_outlier"].round(2)
)

display(summary.sort_values(
    "persentase_outlier", ascending=False
))

,jumlah,kandidat_outlier,persentase_outlier
Currency,,,
Qatari Rial(QR),20,2,10.00
Indonesian Rupiah(IDR),21,2,9.52
Indian Rupees(Rs.),8652,813,9.40
Botswana Pula(P),22,2,9.09
Pounds(��),80,7,8.75
Turkish Lira(TL),34,2,5.88
Emirati Diram(AED),60,3,5.00
NewZealand($),40,2,5.00
Rand(R),60,3,5.00


1. transform() mengembalikan hasil dengan panjang yang sama seperti data awal. Jadi, setiap baris memperoleh kuartil dari kelompok mata uangnya sendiri.
2. lambda s: s.quantile(0.25) merupakan fungsi kecil yang menghitung kuartil pertama dari sebuah kelompok.
3. agg() menjalankan beberapa agregasi sekaligus.
4. s.mean() * 100 menghitung persentase nilai True, karena True diperlakukan sebagai 1 dan False sebagai 0.